## 2 - Run baseline (macro)
In questo esperimento andrò a riprodurre a la baseline (`01_run_baseline_flat.ipynb`) con modelli di ml vanilla,
sul dataset con features aggregate in macro aree.

I dataset processati si trovano nel path `/notebooks/data/processed_macro` e verrà effettuato l'addestramento
per tutti i corsi singolarmente ed infine per il dataset concatenato.

Le metriche di valutazione sono salvate in csv al path `/notebooks/outputs/runs`, compresa la **PR_AUC**
per paragone con i risultati del paper

In [1]:
import os
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

import sys
sys.path.append(os.path.abspath("../.."))
from utils.baseline_helpers import prepare_flattened_data, evaluate_model

import warnings
warnings.filterwarnings("ignore")

In [2]:
INPUT_DIR = "/notebooks/data/processed_macro"
OUTPUT_DIR = "/notebooks/outputs/runs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
CLASSIFIERS = {
    'RandomForest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'LogisticRegression': LogisticRegression(random_state=42, max_iter=1000, n_jobs=-1) 
}

LAGS = [7, 14, 30, 60, 90]

for current_lag in LAGS:
    print(f"[LOG] Starting runs for LAG -> {current_lag}")
    
    results_list = []
    
    for file_name in os.listdir(INPUT_DIR):
        if not file_name.endswith('.csv'):
            continue
            
        file_path = os.path.join(INPUT_DIR, file_name)
        
        # Estraiamo il nome. I file macro si chiamano 'processed_macro_timeseries_1.csv'
        course_name = file_name.replace('processed_macro_', '').replace('.csv', '')
        
        print(f"\n[LOG] Processing course: {course_name} (LAG={current_lag})")
        
        X, y, feature_names = prepare_flattened_data(file_path, lag=current_lag)
        print(f"\n[LOG] Shape input (Flattened): {X.shape}, Shape target: {y.shape}")
        
        for algo_name, model in CLASSIFIERS.items():
            print(f"[LOG] Evaluating {algo_name}...")
            metrics = evaluate_model(algo_name, model, X, y, n_splits=10)
            
            if metrics is not None:
                result_row = {
                    'course': course_name,
                    'algorithm': algo_name,
                    'accuracy': round(metrics['accuracy'], 4),
                    'precision': round(metrics['precision'], 4),
                    'recall': round(metrics['recall'], 4),
                    'f1': round(metrics['f1'], 4),
                    'roc_auc': round(metrics['roc_auc'], 4),
                    'pr_auc': round(metrics['pr_auc'], 4)
                }
                results_list.append(result_row)
                
    if results_list:
        df_results = pd.DataFrame(results_list)
        # NOME FILE AGGIORNATO per distinguerlo dalla run flat
        output_csv = os.path.join(OUTPUT_DIR, f"baseline_macro_lag{current_lag}_metrics.csv")
        df_results.to_csv(output_csv, index=False)
        print(f"\n[SUCCEEDED] Saved results for LAG={current_lag} in {output_csv}")
    else:
        print(f"\n[WARNING] No results generated for LAG={current_lag}")

[LOG] Starting runs for LAG -> 7

[LOG] Processing course: timeseries_6 (LAG=7)

[LOG] Shape input (Flattened): (469, 70), Shape target: (469,)
[LOG] Evaluating RandomForest...


KeyboardInterrupt: 